# MRCD Framework: Triển khai Đầy đủ & Huấn luyện

Notebook này tổng hợp toàn bộ framework Multi-Round Co-Learning Detection (MRCD) vào một tài liệu thực thi duy nhất.

**Cấu trúc:**
1. **Thiết lập Môi trường**: Thư viện và API Keys.
2. **Triển khai Module Cốt lõi**: Xây dựng các Retrievers, Label Generators, và LLM Handlers.
3. **Huấn luyện SLM**: Dựa trên `SLM_2.ipynb`, thực hiện khởi tạo và huấn luyện mô hình RoBERTa.
4. **Pipeline MRCD**: Điều phối quá trình retrieval, selection, và vòng lặp multi-round learning.
5. **Thực thi**: Chạy thử nghiệm framework trên các sự kiện mới nổi.

In [ ]:
!pip install ddgs rank_bm25 wikipedia

In [ ]:
# --- 1. Environment Setup ---
import os
import sys
import json
import re
import random
import math
import warnings
import requests
import io
import matplotlib.pyplot as plt
from abc import ABC, abstractmethod
from typing import Optional, List, Dict

# Suppress warnings
warnings.filterwarnings('ignore')

# Install requirements if running in Colab/Fresh Env
# !pip install google-genai duckduckgo-search rank-bm25 wikipedia transformers torch scikit-learn python-dotenv matplotlib requests

try:
    import torch
    import torch.nn.functional as F
    import numpy as np
    import pandas as pd
    from dotenv import load_dotenv
    from google import genai
    from google.genai import types
    from duckduckgo_search import DDGS
    from rank_bm25 import BM25Okapi
    import wikipedia
    from transformers import (
        RobertaTokenizer, 
        RobertaForSequenceClassification, 
        AdamW, 
        get_linear_schedule_with_warmup
    )
    from torch.utils.data import Dataset, DataLoader
    from sklearn.metrics import accuracy_score, precision_recall_fscore_support
    from tqdm.auto import tqdm
except ImportError as e:
    print(f" Missing library: {e}. Please install necessary packages.")

# Load Environment Variables
load_dotenv()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Using device: {device}")

## 2. Triển khai Các Module Cốt Lõi

### 2.1 Xử lý LLM (LLM Handler)
Module này xử lý các tương tác với Large Language Models (LLM).
- **`BaseLLM`**: Lớp cơ sở trừu tượng cho các LLM.
- **`GeminiLLM`**: Triển khai sử dụng Google Gemini API (yêu cầu `GEMINI_API_KEY`).
- **`MockLLM`**: Triển khai dự phòng để kiểm thử khi không có API key.

In [ ]:
# --- LLM Handler ---
class BaseLLM(ABC):
    @abstractmethod
    def generate_text(self, prompt: str) -> str:
        pass

class MockLLM(BaseLLM):
    def generate_text(self, prompt: str) -> str:
        return "Mock Response: This is a fake news detection result."

class GeminiLLM(BaseLLM):
    def __init__(self, model_name: str = "gemini-2.5-flash", api_key: Optional[str] = None):
        self.model_name = model_name
        _api_key = api_key or os.getenv("GEMINI_API_KEY")
        if not _api_key:
            self.client = None
            print(" GEMINI_API_KEY not found. Helper will return mock responses.")
        else:
            self.client = genai.Client(api_key=_api_key)
    
    def generate_text(self, prompt: str) -> str:
        if not self.client:
             return "Mock Response (No Key)"
        try:
            response = self.client.models.generate_content(
                model=self.model_name,
                contents=types.Part.from_text(text=prompt)
            )
            return response.text if hasattr(response, 'text') else str(response)
        except Exception as e:
            return f"Error calling Gemini API: {str(e)}"

# Global Accessor
_current_llm = None
def get_llm():
    global _current_llm
    if _current_llm is None:
        if os.getenv("GEMINI_API_KEY"):
             _current_llm = GeminiLLM()
             print(" Auto-configured Gemini LLM")
        else:
             _current_llm = MockLLM()
             print(" Auto-configured Mock LLM")
    return _current_llm

### 2.2 Retrievers & Gán nhãn
Các thành phần dùng để truy xuất bằng chứng bên ngoài và tạo các ví dụ few-shot.
- **`search_news`**: Sử dụng DuckDuckGo (đóng vai trò như Bing) để tìm các bài báo liên quan.
- **`retrieve_demonstrations`**: Sử dụng thuật toán BM25 để chọn các ví dụ phù hợp nhất từ kết quả truy xuất để làm ngữ cảnh cho LLM.
- **`generate_label`**: Hàm hỗ trợ để gán nhãn cho các ví dụ đã truy xuất.

In [ ]:
# --- Retrievers: Bing & BM25 & Labels ---

# Label Space Definition
REAL_LABELS = ["Real", "Authentic", "Reliable", "True", "Verified", "Factual", "Credible", "Accurate"]
FAKE_LABELS = ["Fake", "Hoax", "False", "Fabricated", "Misleading", "Unverified", "Dubious", "Deceptive"]
ALL_LABELS = REAL_LABELS + FAKE_LABELS

def generate_label(text):
    """Randomly assigns a label from the defined vocabulary for demonstration purposes."""
    return random.choice(ALL_LABELS)

# News Corpus Handling (Remote CSV)
AG_NEWS_URL = "https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/train.csv"

def load_news_corpus(url=AG_NEWS_URL):
    """Downloads AG News CSV and loads it as a list of text documents."""
    print(f" Downloading News Corpus from {url}...")
    try:
        response = requests.get(url)
        response.raise_for_status()
        
        # Parse CSV (AG News format: Class Index, Title, Description)
        # We only need the text content (Title + Description)
        csv_content = io.StringIO(response.text)
        df = pd.read_csv(csv_content, header=None, names=["class", "title", "desc"])
        
        # Combine title and description
        corpus_texts = (df['title'] + " " + df['desc']).tolist()
        
        print(f" Loaded {len(corpus_texts)} documents from AG News.")
        return corpus_texts
    except Exception as e:
        print(f" Error downloading corpus: {e}")
        return []

# Bing Search Integration
def search_news(query: str, max_results: int = 5):
    try:
        with DDGS() as ddgs:
            results_gen = ddgs.news(query=query, region="wt-wt", safesearch="off", max_results=max_results, backend="bing")
            news_items = []
            for i, result in enumerate(results_gen):
                if i >= max_results: break
                news_items.append(f"{result.get('title', '')}\n{result.get('body', '')}")
            return news_items
    except Exception as e:
        print(f"Bing Search Error: {e}")
        # Fallback Mock Data for Demo if offline
        return [
            f"News Article 1 relating to {query}\nContent body discussing...",
            f"Report on {query}\nDetails emerging..."
        ]

# BM25 Matcher for Demonstration Retrieval
def retrieve_demonstrations(query: str, corpus_items: list, k: int = 4):
    if not corpus_items:
        return []
    tokenized_corpus = [doc.lower().split() for doc in corpus_items]
    bm25 = BM25Okapi(tokenized_corpus)
    scores = bm25.get_scores(query.lower().split())
    
    # Get top K indices
    # Convert numpy types to native Python if needed
    scored_indices = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
    top_k = [idx for idx, s in scored_indices[:k]]
    
    demonstrations = []
    for i in top_k:
        content = corpus_items[i]
        demonstrations.append({
            "text": content,
            "label": generate_label(content),
            "source": "Bing/Retrieved"
        })
    return demonstrations

### 2.3 Wiki Background Agent
Cung cấp xác minh kiến thức nền.
- **`extract_entities`**: Sử dụng LLM để nhận diện các thực thể chính (Người, Tổ chức, Sự kiện) trong tuyên bố.
- **`query_wikipedia`**: Lấy tóm tắt cho các thực thể đã nhận diện để cung cấp ngữ cảnh thực tế.

In [ ]:
# --- Wiki Agent ---
def extract_entities(text: str) -> list:
    prompt = (f"Identify 1-3 key entities (People, Org, Event) in: '{text}'. "
              "Return JSON: [{'entity': 'Name', 'lang': 'en'}]. No markdown.")
    try:
        llm = get_llm()
        resp = llm.generate_text(prompt)
        clean = resp.replace("```json", "").replace("```", "").strip()
        match = re.search(r'\[.*\]', clean, re.DOTALL)
        if match: clean = match.group(0)
        return json.loads(clean)
    except:
        return []

def query_wikipedia(entity: str, lang: str = 'en') -> str:
    try:
        wikipedia.set_lang(lang)
        return wikipedia.summary(entity, sentences=2, auto_suggest=False)
    except:
        return "Not found"

def extract_and_summarize(text: str) -> dict:
    ents = extract_entities(text)
    res = {}
    for e in ents:
        if isinstance(e, dict) and e.get('entity'):
            summ = query_wikipedia(e['entity'], e.get('lang', 'en'))
            if "Not found" not in summ:
                res[e['entity']] = summ
    return res

## 3. Khởi tạo & Huấn luyện SLM (RoBERTa)

### 3.1 Chuẩn bị Dữ liệu (Twitter16)
Tải và xử lý bộ dữ liệu Twitter16 để huấn luyện Small Language Model (SLM).
- **`FakeNewsDataset`**: Lớp Pytorch Dataset để xử lý văn bản và nhãn.
- **`load_data`**: Tải dữ liệu thật từ Twitter16, sắp xếp theo thời gian, và chia tập Train/Test (80/20).
- **`RobertaTokenizer`**: Chuẩn bị văn bản cho mô hình RoBERTa.

In [ ]:
# --- Dataset & Tokenizer ---
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

class FakeNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# --- Load Data (Twitter16) ---
def load_data(data_dir="../data/twitter16"):
    label_path = os.path.join(data_dir, "label.txt")
    tweets_path = os.path.join(data_dir, "source_tweets.txt")
    
    # Fallback to dummy if files missing
    if not os.path.exists(label_path) or not os.path.exists(tweets_path):
        print(f" Data not found at {data_dir}. using Dummy Mode.")
        return generate_dummy_data()

    print(f" Loading real data from {data_dir}...")
    
    # Load Texts
    tweet_texts = {}
    with open(tweets_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                tweet_texts[parts[0]] = parts[1]

    # Load Labels & Merge
    data = []
    with open(label_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split(':')
            if len(parts) >= 2:
                lbl, tid = parts[0], parts[1]
                if tid in tweet_texts:
                    # Map: 0=Real(true, non-rumor), 1=Fake(false, unverified)
                    binary_lbl = 0 if lbl in ['true', 'non-rumor'] else 1
                    data.append({
                        'id': int(tid),
                        'text': tweet_texts[tid],
                        'label': binary_lbl
                    })
    
    # Sort by ID (Chronological)
    data.sort(key=lambda x: x['id'])
    
    # Split 80/20
    split = int(0.8 * len(data))
    train = data[:split]
    test = data[split:]
    
    print(f" Data Processed: {len(data)} total. Split: {len(train)} Train, {len(test)} Test.")
    
    return (
        [x['text'] for x in train], [x['label'] for x in train],
        [x['text'] for x in test], [x['label'] for x in test]
    )

def generate_dummy_data():
    print(" Generating sample training data...")
    texts = [
        "Breaking: Aliens land in New York!", "Study finds water is wet.",
        "Scientists cure all diseases with new pill.", "Market crashes due to unexpected event.",
        "Celebrity announces sudden retirement.", "Mars colony established by 2025.",
        "New tax law passed by congress.", "Local cat writes shakespeare play.",
        "Weather forecast predicts sunny days.", "Chocolate helps you lose weight fast."
    ] * 10
    labels = [1, 0, 1, 0, 0, 1, 0, 1, 0, 1] * 10
    return texts, labels, [], []

# Initialize Tokenizer
print(" Loading Tokenizer...")
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

# Prepare Data
train_texts, train_labels, test_texts, test_labels = load_data()
train_dataset = FakeNewsDataset(train_texts, train_labels, tokenizer)
# UPDATED: Batch Size = 32
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True) 
print(f" Train Loader prepared with {len(train_dataset)} samples.")

### 3.2 Vòng lặp Huấn luyện SLM
Fine-tune mô hình **RoBERTa-base** cho tác vụ phát hiện tin giả cụ thể.
- **Optimizer**: AdamW (Learning Rate: 1e-3, Weight Decay: 1e-4)
- **Batch Size**: 32
- **Quy trình**: Huấn luyện trong 2 epochs và lưu mô hình vào `model_SLM_twitter16`.
- **Trình tối ưu hóa (Optimizer)**: Sử dụng AdamW.
- **Tốc độ học (Learning rate)**: Bắt đầu ở mức 1e-3.
- **Suy giảm trọng số (Weight decay)**: 1e-4.

In [ ]:
# --- Model Definition & Training Loop ---
print(" Loading RoBERTa Model...")
model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=2)
model.to(device)

# UPDATED Hyperparameters: LR=1e-3, Weight Decay=1e-4
optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
epochs = 3 # Increased slightly to see curve better
total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

def train_slm():
    model.train()
    
    # Tracking Metrics
    history = {
        'loss': [],
        'accuracy': []
    }
    
    for epoch in range(epochs):
        total_loss = 0
        all_preds = []
        all_labels = []
        
        print(f"\n Epoch {epoch+1}/{epochs}")
        progress_bar = tqdm(train_loader, desc="Training")
        
        for batch in progress_bar:
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            total_loss += loss.item()
            
            # Save predictions for accuracy calculation
            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            loss.backward()
            optimizer.step()
            scheduler.step()
            
            progress_bar.set_postfix({'loss': loss.item()})
            
        # Calculate Epoch Metrics
        avg_loss = total_loss / len(train_loader)
        epoch_acc = accuracy_score(all_labels, all_preds)
        
        history['loss'].append(avg_loss)
        history['accuracy'].append(epoch_acc)
        
        print(f"   Loss: {avg_loss:.4f} | Accuracy: {epoch_acc:.4f}")

    print("\n Training Complete.")
    
    # Plotting
    plt.figure(figsize=(12, 5))
    
    # Loss Plot
    plt.subplot(1, 2, 1)
    plt.plot(history['loss'], label='Training Loss', color='red', marker='o')
    plt.title('Training Loss per Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    
    # Accuracy Plot
    plt.subplot(1, 2, 2)
    plt.plot(history['accuracy'], label='Training Accuracy', color='blue', marker='o')
    plt.title('Training Accuracy per Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()
    
    # Save Model
    save_path = "model_SLM_twitter16"
    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)
    print(f" Model saved to {save_path}")

# Execute Training
train_slm()

# --- Evaluation on Test Set (Immediately after Init) ---
print("\n--- Evaluating Initial SLM on Test Set ---")
# Create Test Loader
test_dataset = FakeNewsDataset(test_texts, test_labels, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Calculate Metrics
acc = accuracy_score(all_labels, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='binary')

print(f" Test Accuracy:  {acc:.4f}")
print(f" Test Precision: {precision:.4f}")
print(f" Test Recall:    {recall:.4f}")
print(f" Test F1 Score:  {f1:.4f}")

### 3.3 Integrated SLM Wrapper
Một lớp thống nhất để quản lý vòng đời của SLM trong pipeline.
- **`inference`**: Trả về dự đoán (Real/Fake) và điểm tin cậy (confidence score).
- **`train`**: Cho phép huấn luyện lại trực tuyến (Co-Learning) trên các mẫu có độ tin cậy cao được xác định trong quá trình chạy pipeline.

In [ ]:
# --- Integrated SLM Wrapper ---
class IntegratedSLM:
    def __init__(self, model_path='model_SLM_twitter16'):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        if not os.path.exists(model_path):
             print(" Saved model not found. Using untrained base.")
             model_path = "roberta-base"
        else:
             print(f" Loading SLM from {model_path}")
             
        self.tokenizer = RobertaTokenizer.from_pretrained(model_path)
        self.model = RobertaForSequenceClassification.from_pretrained(model_path, num_labels=2)
        self.model.to(self.device)
        self.model.eval()

    def inference(self, text):
        text = text.lower() # Simple preprocess
        inputs = self.tokenizer(text, return_tensors='pt', truncation=True, max_length=128, padding='max_length')
        with torch.no_grad():
            outputs = self.model(inputs['input_ids'].to(self.device), inputs['attention_mask'].to(self.device))
            probs = F.softmax(outputs.logits, dim=1)
        conf, pred = torch.max(probs, dim=1)
        return pred.item(), conf.item(), probs[0].cpu().numpy()
        
    def train(self, d_clean: list, epochs: int=1):
        # Logic to fine-tune on new clean data
        if not d_clean: return
        print(f"    Retraining SLM on {len(d_clean)} new clean samples...")
        texts = [d['text'] for d in d_clean]
        labels = [d['label'] for d in d_clean]
        dataset = FakeNewsDataset(texts, labels, self.tokenizer)
        # UPDATED: Batch Size = 32
        loader = DataLoader(dataset, batch_size=32, shuffle=True)
        
        self.model.train()
        # UPDATED: LR=1e-3, Weight Decay=1e-4
        optim = AdamW(self.model.parameters(), lr=1e-3, weight_decay=1e-4)
        for _ in range(epochs):
            for batch in loader:
                 optim.zero_grad()
                 outputs = self.model(batch['input_ids'].to(self.device), 
                                      batch['attention_mask'].to(self.device), 
                                      labels=batch['labels'].to(self.device))
                 outputs.loss.backward()
                 optim.step()
        self.model.eval()
        print("    Retraining Complete.")

# Initialize Wrapper
slm_instance = IntegratedSLM()

## 4. Triển khai Pipeline MRCD

### 4.1 Logic Pipeline MRCD
Sự điều phối cốt lõi của framework Multi-Round Co-Learning Detection.
1.  **Retrieval**: Lấy thông tin News (Bing) và kiến thức nền (Wiki).
2.  **Assessment (Vòng 1)**: Lấy đánh giá ban đầu từ LLM và SLM.
3.  **Selection**: Tách các mẫu thành **Clean** (Đồng thuận cao) và **Noisy** (Bất đồng/Độ tin cậy thấp).
4.  **Co-Learning (Vòng 2+)**: Huấn luyện lại SLM trên dữ liệu 'Clean' và đánh giá lại dữ liệu 'Noisy' để giải quyết sự mơ hồ.

In [ ]:
CONFIDENCE_THRESHOLD = 0.8

def run_mrcd_pipeline(events: List[str]):
    print(f" Starting MRCD Pipeline for {len(events)} events...")
    
    # Connect to LLM
    llm = get_llm()
    
    # Load Static Corpus for Retrieval (Context/Demos)
    print(" Loading News Corpus (Fact-Checking Base)...")
    static_corpus = load_news_corpus() 
    print(f" Corpus Loaded: {len(static_corpus)} documents.")
    
    d_clean = []
    d_noisy = []
    knowledge_cache = {}
    
    # --- Round 1 ---
    print("\n=== Step 1 & 2: Initial Retrieval & Selection ===")
    for text in tqdm(events, desc="Processing Events"):
        # print(f"\n Processing: '{text[:50]}...'")
        
        # 1. Retrieval
        query = text 
        
        # Live Search (Current Evidence)
        bing_res = search_news(query, 3)
        
        # Demonstration Retrieval (Few-Shot)
        # Combine static corpus with fresh results to find most relevant similar cases
        # Note: Efficiency can be improved by pre-indexing static corpus
        combined_corpus = static_corpus + bing_res
        demos = retrieve_demonstrations(query, combined_corpus, k=3)
        
        # Background Knowledge
        wiki_res = extract_and_summarize(text)
        knowledge_k = "\n".join([f"[{k}]: {v}" for k,v in wiki_res.items()]) if wiki_res else "No info."
        knowledge_cache[text] = knowledge_k
        
        # 2. LLM Pred
        prompt = f"Fact Check:\nInfo: {knowledge_k}\nNews: {text}\nExamples: {demos}\nReply 0 (Real) or 1 (Fake) only."
        llm_resp = llm.generate_text(prompt).strip().lower()
        y_llm = 0 if ('0' in llm_resp or 'real' in llm_resp) else 1
        
        # 3. SLM Pred
        y_slm, conf_slm, _ = slm_instance.inference(text)
        
        sample = {"text": text, "label": y_llm, "conf": conf_slm}
        
        if y_llm == y_slm and conf_slm >= CONFIDENCE_THRESHOLD:
            d_clean.append(sample)
            # print(f"    Clean (Label: {y_llm}, Conf: {conf_slm:.2f})")
        else:
            d_noisy.append(sample)
            # print(f"    Noisy (LLM:{y_llm} != SLM:{y_slm} or Conf:{conf_slm:.2f} too low)")

    # --- Round 2+ (Co-Learning) ---
    if d_noisy:
        print(f"\n=== Step 3: Multi-Round Co-Learning (Noisy: {len(d_noisy)}) ===")
        # Train SLM on D_clean
        slm_instance.train(d_clean)
        
        # Re-evaluate Noisy
        final_results = d_clean + []
        for sample in d_noisy:
            text = sample['text']
            # Re-infer
            y_slm, conf, _ = slm_instance.inference(text)
            # For Demo: Force Final Judgment
            sample['label'] = y_slm
            sample['final_conf'] = conf
            sample['status'] = "Resolved via Co-Learning"
            final_results.append(sample)
            # print(f"    Resolved '{text[:30]}...' -> {y_slm} (Conf: {conf:.2f})")
            
        return final_results
    else:
        return d_clean

## 5. Thực thi Demo & Đánh giá

In [ ]:
# --- Evaluation on Test Set ---
if len(test_texts) > 0:
    print(f" Evaluation on {len(test_texts)} Test Samples...")
    
    # Run Pipeline on Test Subset (Taking first 20 for demo speed, remove slice for full eval)
    # Note: Running full test set might take time due to API calls (Bing/Gemini)
    TEST_LIMIT = 20 
    eval_texts = test_texts[:TEST_LIMIT]
    eval_labels = test_labels[:TEST_LIMIT]
    
    # Use model inference directly for pure SLM eval, or pipeline for full system eval
    # Here we show SLM metrics (since 'pipeline' is complex co-learning)
    
    print(" Evaluating Integrated SLM performance on Test Set...")
    slm_preds = []
    
    for text in tqdm(eval_texts):
        pred, _, _ = slm_instance.inference(text)
        slm_preds.append(pred)
        
    acc = accuracy_score(eval_labels, slm_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(eval_labels, slm_preds, average='binary')
    
    print("\n Final Evaluation Metrics (SLM Only):")
    print(f" Accuracy:  {acc:.4f}")
    print(f" Precision: {precision:.4f}")
    print(f" Recall:    {recall:.4f}")
    print(f" F1 Score:  {f1:.4f}")
    
    # Optional: Run Full Pipeline on a few samples to demo Co-Learning
    print("\n Running Full MRCD Pipeline (Demo on 5 samples)...")
    results = run_mrcd_pipeline(eval_texts[:5])
    for i, r in enumerate(results):
         truth = "Fake" if eval_labels[i]==1 else "Real"
         pred = "Fake" if r['label']==1 else "Real"
         print(f" [{truth} -> {pred}] {r['text'][:50]}...")

else:
    print(" No test data available.")